# How to configure the neural network

A **config object** describes the neural estimator and its settings. Pass it to a trainer to choose and configure the network.

There is one config class per model, and it carries only the settings that model actually accepts. A setting the model does not have is therefore not a field on its config, so passing it raises an error right away instead of being silently ignored.

This guide covers:

- picking a model and changing its hyperparameters
- adding an embedding network and controlling z-scoring
- the configs for classifiers, mixed data, marginal densities, and vector fields
- migrating from strings and factory functions

## Setup

In [ ]:
import torch

from sbi.utils import BoxUniform

num_dim = 2
prior = BoxUniform(low=-2 * torch.ones(num_dim), high=2 * torch.ones(num_dim))

theta = prior.sample((500,))
x = theta + 1.0 + 0.1 * torch.randn_like(theta)

## Choosing a density estimator

`NPE` and `NLE` take their network as `density_estimator`. Import the config of the model you want and pass an instance:

In [ ]:
from sbi.inference import NPE
from sbi.neural_nets import ZukoNSFConfig

trainer = NPE(prior=prior, density_estimator=ZukoNSFConfig())

Hyperparameters are constructor arguments, so your editor can complete them and type checkers can see them:

In [ ]:
config = ZukoNSFConfig(hidden_features=64, num_transforms=8, num_bins=12)
trainer = NPE(prior=prior, density_estimator=config)

The available density estimator configs are:

| Config | Model |
|---|---|
| `MAFConfig` | masked autoregressive flow (`nflows`) |
| `MAFRQSConfig` | MAF with rational-quadratic splines (`nflows`) |
| `NSFConfig` | neural spline flow (`nflows`) |
| `MADEConfig` | masked autoencoder for density estimation |
| `MDNConfig` | mixture density network |
| `ZukoMAFConfig`, `ZukoNSFConfig`, `ZukoNCSFConfig`, `ZukoNAFConfig`, `ZukoUNAFConfig`, `ZukoBPFConfig`, `ZukoSOSPFConfig`, `ZukoNICEConfig`, `ZukoGFConfig` | the corresponding [`zuko`](https://github.com/probabilists/zuko) flows |

For model recommendations, see [how to choose neural nets](03_choose_neural_net.ipynb). `NPE_A` accepts `MDNConfig` only. The pretrained `TabPFNConfig` is used with `NPE_PFN`, not with `NPE` or `NLE`.

`NLE` estimates the likelihood rather than the posterior, but it takes the same configs, because the difference is which variable is modeled and which is conditioned on. That is decided by the trainer, not by the config:

In [ ]:
from sbi.inference import NLE
from sbi.neural_nets import MAFConfig

trainer = NLE(prior=prior, density_estimator=MAFConfig(hidden_features=64))

## Settings a model does not have

`num_bins` is a spline setting. `NSFConfig` has it, `MAFConfig` does not, so the mistake surfaces immediately:

In [ ]:
from sbi.neural_nets import NSFConfig

NSFConfig(num_bins=12)  # fine, a spline flow has bins

try:
    MAFConfig(num_bins=12)
except TypeError as e:
    print(e)

The same applies to a misspelled setting, and to a value outside the allowed set:

In [ ]:
try:
    NSFConfig(hiden_features=64)
except TypeError as e:
    print(e)

try:
    NSFConfig(z_score_input="strucured")
except ValueError as e:
    print(e)

## Inspecting a config

Configs are frozen dataclasses, and their `repr` shows only what you changed, which makes them convenient to log or to put in an experiment record:

In [ ]:
print(ZukoNSFConfig())
print(ZukoNSFConfig(hidden_features=64, num_transforms=8))

Config fields cannot be reassigned after construction, though contained objects such as embedding networks remain mutable. To vary one setting, build a new config:

In [ ]:
configs = [ZukoNSFConfig(hidden_features=h) for h in (32, 64, 128)]
print(configs)

## Embedding networks

For high-dimensional data, pass an embedding network that learns summary statistics of the conditioning variable. See [how to use embedding networks](04_embedding_networks.ipynb) for how to choose one.

In [ ]:
from sbi.neural_nets.embedding_nets import FCEmbedding

config = ZukoNSFConfig(embedding_net=FCEmbedding(input_dim=num_dim, output_dim=8))
trainer = NPE(prior=prior, density_estimator=config)

## Z-scoring

For the trainable conditional density configs, both variables are z-scored independently by default. The standardization is built into the estimator, so you keep passing raw data at inference time.

`z_score_input` applies to the modeled variable and `z_score_condition` to the variable conditioned on. Which is which depends on the method: for `NPE` the input is $\theta$ and the condition is $x$, for `NLE` it is the other way round. Each takes `"independent"` (default), `"structured"`, or `"none"`:

In [ ]:
config = ZukoNSFConfig(z_score_input="independent", z_score_condition="structured")
trainer = NPE(prior=prior, density_estimator=config)

`"independent"` computes separate statistics for each coordinate across the batch. `"structured"` uses a shared mean and scale across coordinates, for example to preserve relative values within a time series or image. Use `"none"` to disable standardization.

Some density configs also support `z_score_input="transform_to_unconstrained"` for bounded inputs; see the [unconstrained-space FAQ](../faq/question_04_unconstrained.md).

## Settings without a field

Some underlying models accept keyword arguments that have no field on the config. Use `extra_kwargs` for these library-specific options. The underlying builder must support the option; this is not a way to make an unsupported setting work:

In [ ]:
config = ZukoNSFConfig(hidden_features=64, extra_kwargs={"randperm": True})
trainer = NPE(prior=prior, density_estimator=config)

A key that duplicates an existing field is rejected, so there is one place a setting can come from:

In [ ]:
try:
    ZukoNSFConfig(extra_kwargs={"hidden_features": 64})
except ValueError as e:
    print(e)

## Classifiers for NRE

The `NRE` variants train a classifier instead of a density estimator, and take it as `classifier`:

In [ ]:
from sbi.inference import NRE
from sbi.neural_nets import ResNetClassifierConfig

config = ResNetClassifierConfig(hidden_features=64, num_blocks=3)
trainer = NRE(prior=prior, classifier=config)

The classifier configs are `LinearClassifierConfig`, `MLPClassifierConfig`, and `ResNetClassifierConfig`. A classifier sees both variables as inputs rather than conditioning on one, so it takes two embedding networks instead of one:

In [ ]:
config = ResNetClassifierConfig(
    embedding_net_theta=FCEmbedding(input_dim=num_dim, output_dim=8),
    embedding_net_x=FCEmbedding(input_dim=num_dim, output_dim=8),
)

## Mixed data for MNPE and MNLE

`MNPE` and `MNLE` handle data that is partly continuous and partly discrete. `MixedConfig` describes the whole estimator, and the continuous part is configured by nesting that model's own config, so its settings stay validated by its own class:

In [ ]:
from sbi.inference import MNLE
from sbi.neural_nets import MixedConfig

config = MixedConfig(
    continuous=ZukoNSFConfig(hidden_features=64, num_transforms=4),
    discrete_hidden_features=32,
)
trainer = MNLE(prior=prior, density_estimator=config)

The modeled variable is $x$ for MNLE and $\theta$ for MNPE. Put its continuous columns first and discrete columns last. `MixedConfig()` defaults to an nflows NSF continuous component with `tail_bound=10.0`.

Set input z-scoring on `continuous.z_score_input`. Set condition z-scoring and the condition embedding on `MixedConfig`, not on its nested continuous config. Put library-specific continuous options in `continuous.extra_kwargs`; top-level `MixedConfig.extra_kwargs` is unsupported.

## Marginal densities

`MarginalTrainer` fits an unconditional density, so its configs have no conditioning variable and no `z_score_condition`:

In [ ]:
from sbi.inference.trainers.marginal import MarginalTrainer
from sbi.neural_nets import MarginalNSFConfig

trainer = MarginalTrainer(
    density_estimator=MarginalNSFConfig(hidden_features=64, num_transforms=4)
)

## Vector fields for FMPE and NPSE

Pass an estimator config as `vf_estimator`. `FlowMatchingConfig` is for FMPE; `VEScoreConfig`, `VPScoreConfig`, and `SubVPScoreConfig` select NPSE's SDE. Choose the architecture separately through `net`:

In [ ]:
from sbi.inference import FMPE, NPSE
from sbi.neural_nets import (
    FlowMatchingConfig,
    MLPConfig,
    TransformerConfig,
    VPScoreConfig,
)

flow_config = FlowMatchingConfig(net=MLPConfig(hidden_features=64, num_layers=3))
flow_trainer = FMPE(prior=prior, vf_estimator=flow_config)

score_config = VPScoreConfig(
    net=TransformerConfig(hidden_features=64, num_heads=4), beta_max=20.0
)
score_trainer = NPSE(prior=prior, vf_estimator=score_config)

The network configs are `MLPConfig`, `AdaMLPConfig`, and `TransformerConfig`. `TransformerConfig(is_x_emb_seq=True)` selects cross-attention for sequence conditions. Both `FlowMatchingConfig()` and the default `VEScoreConfig()` use `MLPConfig()`.

Keep `embedding_net`, `z_score_input`, and `z_score_condition` on the estimator config. Estimator options without fields go in its `extra_kwargs`; network options go in `net.extra_kwargs`. Do not pass `sde_type` to NPSE together with a config: the config class already selects it.

For custom networks, pass a `VectorFieldNet` instance as `net`. See the [vector-field tutorial](../advanced_tutorials/19_vector_field_methods.ipynb) for examples and noise schedules.

## Building the estimator yourself

The trainer calls `build` on the config once it has seen data, because the network needs the shapes and the z-scoring statistics. You can call it yourself if you want the network outside a trainer, for example for a custom training loop as in [the training interface tutorial](../advanced_tutorials/18_training_interface.ipynb):

In [ ]:
estimator = ZukoNSFConfig(hidden_features=32).build(theta, x)
print(type(estimator).__name__, "| input shape", estimator.input_shape)

The first argument is the modeled variable and the second the one conditioned on, matching `z_score_input` and `z_score_condition`. Marginal configs take only one batch: `MarginalNSFConfig().build(x)`.

For a custom estimator, pass a build function to the trainer. Unlike `config.build`, the trainer calls this function with `(batch_theta, batch_x)` for both NPE and NLE. Return an estimator implementing the required interface, not an arbitrary `torch.nn.Module`:

In [ ]:
def build_likelihood(batch_theta, batch_x):
    return MAFConfig(hidden_features=32).build(
        batch_input=batch_x, batch_condition=batch_theta
    )


trainer = NLE(prior=prior, density_estimator=build_likelihood)

## Moving off strings and factory functions

Before the configs, the network was selected with a string, or with one of the `posterior_nn` / `likelihood_nn` / `classifier_nn` factory functions. Both still work; passing a string now warns.

| Before | Now |
|---|---|
| `NPE(prior, density_estimator="nsf")` | `NPE(prior, density_estimator=NSFConfig())` |
| `NPE(prior, density_estimator="zuko_nsf")` | `NPE(prior, density_estimator=ZukoNSFConfig())` |
| `NLE(prior, density_estimator="maf")` | `NLE(prior, density_estimator=MAFConfig())` |
| `NRE(prior, classifier="resnet")` | `NRE(prior, classifier=ResNetClassifierConfig())` |
| `posterior_nn(model="nsf", hidden_features=64)` | `NSFConfig(hidden_features=64)` |
| `likelihood_nn(model="maf", num_transforms=8)` | `MAFConfig(num_transforms=8)` |
| `classifier_nn(model="mlp", hidden_features=64)` | `MLPClassifierConfig(hidden_features=64)` |
| `MNLE(prior, density_estimator="mnle")` | `MNLE(prior, density_estimator=MixedConfig())` |
| `MNPE(prior, density_estimator="mnpe")` | `MNPE(prior, density_estimator=MixedConfig())` |
| `MarginalTrainer(density_estimator=ZukoFlowType.NSF)` | `MarginalTrainer(density_estimator=MarginalNSFConfig())` |
| `posterior_flow_nn(model="mlp", hidden_features=64)` | `FlowMatchingConfig(net=MLPConfig(hidden_features=64))` |
| `posterior_score_nn(model="mlp", sde_type="vp")` | `VPScoreConfig(net=MLPConfig())` |

Most model-specific settings keep their names, but z-scoring names depend on the variable's role:

| Factory argument | Config field |
|---|---|
| `posterior_nn(z_score_theta=...)` | `z_score_input` |
| `posterior_nn(z_score_x=...)` | `z_score_condition` |
| `likelihood_nn(z_score_x=...)` | `z_score_input` |
| `likelihood_nn(z_score_theta=...)` | `z_score_condition` |
| `classifier_nn(z_score_theta=..., z_score_x=...)` | `z_score_input`, `z_score_condition` |
| `marginal_nn(z_score_x=...)` | `z_score_input` |
| `posterior_flow_nn` / `posterior_score_nn`: `z_score_theta`, `z_score_x` | `z_score_input`, `z_score_condition` |
| VF factories: `t_embedding_dim` | `net.time_embedding_dim` |

For marginal spline configs, use `bins` instead of the factory's `num_bins`. Use `"none"` to disable z-scoring in configs rather than the legacy `None`.

In [ ]:
from sbi.neural_nets import posterior_nn

# Before
build_fn = posterior_nn(model="zuko_nsf", hidden_features=64, num_transforms=8)
trainer = NPE(prior=prior, density_estimator=build_fn)

# Now
config = ZukoNSFConfig(hidden_features=64, num_transforms=8)
trainer = NPE(prior=prior, density_estimator=config)

Configs reject misspelled or inapplicable fields at construction. The density, classifier, mixed, and vector-field factories now also reject supplied non-default settings that their selected model does not use. Genuinely unknown factory keywords still warn and are forwarded for compatibility. With configs, use `extra_kwargs` only for options supported by the underlying library.

## See also

- [How to choose neural nets](03_choose_neural_net.ipynb) for which model to pick
- [How to use embedding networks](04_embedding_networks.ipynb)
- [How to choose abstraction levels](24_abstraction_levels.ipynb) for where configs sit relative to the other ways of specifying a network